# Token Embeddings for LLMs

This notebook builds intuition for token embeddings and then demonstrates how a GPT-style embedding matrix is created, queried, and updated by backpropagation. It uses small examples and stops before positional embeddings, attention, or a Transformer.

## 1. Why token IDs are not enough

A tokenizer can assign an integer ID to every token, but the number is only an identifier. `dog = 0`, `cat = 1`, and `banana = 3` does **not** mean dog is semantically closer to cat than banana. Arithmetic distance between arbitrary IDs has no linguistic meaning.

One-hot vectors make tokens distinct without suggesting an ordering, but all different one-hot vectors are equally dissimilar. They still contain no learned semantic relationship.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

words = ["dog", "cat", "apple", "banana"]
word_to_id = {word: token_id for token_id, word in enumerate(words)}
one_hot_vectors = F.one_hot(
    torch.tensor(list(word_to_id.values())),
    num_classes=len(words),
).float()

print("Token IDs:")
for word, token_id in word_to_id.items():
    print(f"{word:<6} -> {token_id}")

print("\nOne-hot vectors:")
for word, vector in zip(words, one_hot_vectors):
    print(f"{word:<6} -> {vector.tolist()}")

Token IDs:
dog    -> 0
cat    -> 1
apple  -> 2
banana -> 3

One-hot vectors:
dog    -> [1.0, 0.0, 0.0, 0.0]
cat    -> [0.0, 1.0, 0.0, 0.0]
apple  -> [0.0, 0.0, 1.0, 0.0]
banana -> [0.0, 0.0, 0.0, 1.0]


Every pair of different one-hot vectors above has the same dot product, zero. Nothing in those vectors says that `dog` and `cat` are related.

## 2. An intuitive semantic-vector example

Before using learned embeddings, we can manually invent understandable dimensions. These are teaching features—not how GPT embeddings are designed.

In [2]:
features = [
    "has_tail",
    "is_eatable",
    "has_four_legs",
    "makes_sound",
    "is_pet",
]

semantic_vectors = {
    "dog":    torch.tensor([1.0, 0.0, 1.0, 1.0, 1.0]),
    "cat":    torch.tensor([1.0, 0.0, 1.0, 1.0, 1.0]),
    "apple":  torch.tensor([0.0, 1.0, 0.0, 0.0, 0.0]),
    "banana": torch.tensor([0.0, 1.0, 0.0, 0.0, 0.0]),
}

print("Feature order:", features)
for word, vector in semantic_vectors.items():
    print(f"{word:<6} -> {vector.tolist()}")

Feature order: ['has_tail', 'is_eatable', 'has_four_legs', 'makes_sound', 'is_pet']
dog    -> [1.0, 0.0, 1.0, 1.0, 1.0]
cat    -> [1.0, 0.0, 1.0, 1.0, 1.0]
apple  -> [0.0, 1.0, 0.0, 0.0, 0.0]
banana -> [0.0, 1.0, 0.0, 0.0, 0.0]


Cosine similarity compares vector direction. A value near 1 means similar directions, while a value near 0 means little directional similarity in this non-negative toy example.

In [3]:
def cosine_similarity(vector_a, vector_b):
    return F.cosine_similarity(
        vector_a.unsqueeze(0),
        vector_b.unsqueeze(0),
    ).item()

manual_pairs = [
    ("dog", "cat"),
    ("apple", "banana"),
    ("dog", "banana"),
]

for first_word, second_word in manual_pairs:
    score = cosine_similarity(
        semantic_vectors[first_word],
        semantic_vectors[second_word],
    )
    print(f"{first_word:<6} vs {second_word:<6} -> cosine similarity {score:.3f}")

dog    vs cat    -> cosine similarity 1.000
apple  vs banana -> cosine similarity 1.000
dog    vs banana -> cosine similarity 0.000


The manually designed vectors make `dog` similar to `cat` and `apple` similar to `banana`, while separating `dog` from `banana`. Vectors can therefore represent relationships that arbitrary IDs and one-hot vectors do not naturally express. Real embeddings learn their dimensions from data instead of using these hand-written features.

## 3. Pretrained word-vector demonstration with Gensim

This optional section uses pretrained **static word vectors** to show that vector spaces can contain useful semantic relationships. The preferred model is `glove-wiki-gigaword-50`, a much lighter download than `word2vec-google-news-300`.

Use the registered **Python 3.11 (LLM + Gensim)** notebook kernel for this section. Gensim 4.4.0 has no compatible prebuilt Windows wheel for the Python 3.14 kernel used previously, so Python 3.14 attempted—and failed—to compile Gensim from source. The rest of the notebook does not require Gensim or a model download.

In [4]:
import sys

print("Notebook kernel Python:", sys.executable)
print("Python version:", sys.version.split()[0])

if sys.version_info >= (3, 14):
    print("Switch the notebook kernel to: Python 3.11 (LLM + Gensim)")
else:
    print("This Python version can use the installed Gensim package.")

# Only if Gensim is missing from a Python 3.11/3.12 kernel:
# %pip install --upgrade gensim

Notebook kernel Python: c:\Users\user\AppData\Local\Programs\Python\Python311\python.exe
Python version: 3.11.9
This Python version can use the installed Gensim package.


In [5]:
PRETRAINED_MODEL_NAME = "glove-wiki-gigaword-50"
# Optional lecture model; it is a much larger download:
# PRETRAINED_MODEL_NAME = "word2vec-google-news-300"

LOAD_PRETRAINED_MODEL = True
pretrained_model = None

try:
    import gensim
    import gensim.downloader as gensim_api
except Exception as error:
    print("Gensim could not be imported by this notebook kernel.")
    print("Notebook Python:", sys.executable)
    print("Error type:", type(error).__name__)
    print("Full error:", repr(error))
    print("Run `%pip install --upgrade gensim` in a notebook cell, then restart the kernel.")
else:
    print("Gensim version:", gensim.__version__)
    if LOAD_PRETRAINED_MODEL:
        try:
            print(f"Loading {PRETRAINED_MODEL_NAME!r}...")
            pretrained_model = gensim_api.load(PRETRAINED_MODEL_NAME)
            print("Loaded vector size:", pretrained_model.vector_size)
        except Exception as error:
            print("The pretrained model could not be loaded.")
            print("Check the network connection and rerun this cell.")
            print("Error:", error)
    else:
        print("Set LOAD_PRETRAINED_MODEL = True to run the Gensim demonstrations.")

Gensim version: 4.4.0
Loading 'glove-wiki-gigaword-50'...
[==================================================] 100.0% 66.0/66.0MB downloaded
Loaded vector size: 50


### Vector analogy: king + woman - man

The expression asks for vectors near the direction produced by adding `woman` to `king` and subtracting `man`. Results vary by pretrained model and training data.

In [24]:
if pretrained_model is None:
    print("Pretrained analogy skipped until the Gensim model is loaded.")
else:
    analogy_results = pretrained_model.most_similar(
        positive=["winter", "cold"],
        negative=["hot"],
        topn=10,
    )
    for word, score in analogy_results:
        print(f"{word:<15} {score:.4f}")

summer          0.7501
autumn          0.7322
spring          0.7169
during          0.6813
drought         0.6787
rainy           0.6729
war             0.6675
warmer          0.6587
beginning       0.6491
weather         0.6370


### Similarities and nearest words

In [25]:
pretrained_pairs = [
    ("woman", "man"),
    ("king", "queen"),
    ("uncle", "aunt"),
    ("boy", "girl"),
    ("nephew", "niece"),
    ("paper", "water"),
]

if pretrained_model is None:
    print("Pretrained similarities skipped until the Gensim model is loaded.")
else:
    for first_word, second_word in pretrained_pairs:
        score = pretrained_model.similarity(first_word, second_word)
        print(f"{first_word:<14} {second_word:<14} cosine={score:.4f}")

    print("\nNearest words to 'tower':")
    for word, score in pretrained_model.most_similar("tower", topn=10):
        print(f"{word:<15} {score:.4f}")

woman          man            cosine=0.8860
king           queen          cosine=0.7839
uncle          aunt           cosine=0.7631
boy            girl           cosine=0.9327
nephew         niece          cosine=0.8011
paper          water          cosine=0.5507

Nearest words to 'tower':
towers          0.8771
gate            0.7934
building        0.7873
built           0.7804
roof            0.7781
skyscraper      0.7488
constructed     0.7471
dome            0.7415
facade          0.7310
entrance        0.7280


### Euclidean vector distances

A smaller Euclidean distance often suggests more similar representations; a larger distance suggests less similarity. This is only intuition: cosine similarity is commonly used when the direction of embedding vectors is the focus.

In [26]:
import numpy as np

distance_pairs = [
    ("man", "woman"),
    ("nephew", "niece"),
    ("semiconductor", "earth"),
]

if pretrained_model is None:
    print("Pretrained distances skipped until the Gensim model is loaded.")
else:
    for first_word, second_word in distance_pairs:
        missing = [
            word for word in (first_word, second_word)
            if word not in pretrained_model.key_to_index
        ]
        if missing:
            print(f"{first_word} - {second_word}: unavailable; missing {missing}")
            continue
        distance = np.linalg.norm(
            pretrained_model[first_word] - pretrained_model[second_word]
        )
        print(f"{first_word:<14} {second_word:<14} distance={distance:.4f}")

man            woman          distance=2.6026
nephew         niece          distance=2.7750
semiconductor  earth          distance=6.8545


The pretrained Gensim vectors above are useful for intuition, but **GPT does not normally take Word2Vec or GloVe vectors and use them directly**. A GPT-style token embedding matrix is part of the language model and is trained jointly with all its other parameters.

## 4. How GPT-style token embeddings are created

We will use six tokens and three learned numbers per token. The embedding starts with random trainable weights.

In [9]:
tiny_vocabulary = {
    "fox": 0,
    "house": 1,
    "in": 2,
    "is": 3,
    "quick": 4,
    "the": 5,
}
vocab_size = 6
embedding_dim = 3

torch.manual_seed(123)
embedding = nn.Embedding(vocab_size, embedding_dim)

print("Vocabulary:", tiny_vocabulary)
print("\nEmbedding weights:")
print(embedding.weight)
print("\nMatrix shape:", embedding.weight.shape)

Vocabulary: {'fox': 0, 'house': 1, 'in': 2, 'is': 3, 'quick': 4, 'the': 5}

Embedding weights:
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)

Matrix shape: torch.Size([6, 3])


The shape is `6 rows x 3 columns`:

- One row belongs to one token ID.
- That row contains the token's three-dimensional embedding vector.
- Every displayed number is currently a random, trainable model parameter.

## 5. Embedding lookup

Passing an ID to `nn.Embedding` retrieves the corresponding matrix row.

In [27]:
token_id = 3
looked_up_vector = embedding(torch.tensor(token_id))
direct_matrix_row = embedding.weight[token_id]

print("Token ID:", token_id)
print("Token:   ", next(word for word, index in tiny_vocabulary.items() if index == token_id))
print("Embedding lookup:", looked_up_vector)
print("Direct row:      ", direct_matrix_row)
print("Identical:", torch.equal(looked_up_vector, direct_matrix_row))

Token ID: 3
Token:    is
Embedding lookup: tensor([-0.4015,  0.9666, -1.1481], grad_fn=<EmbeddingBackward0>)
Direct row:       tensor([-0.4015,  0.9666, -1.1481], grad_fn=<SelectBackward0>)
Identical: True


`nn.Embedding` is essentially a learnable lookup table. Multiple IDs retrieve multiple rows in the same order.

In [28]:
input_ids = torch.tensor([2, 3, 5, 1])
lookup_result = embedding(input_ids)
selected_rows = embedding.weight[input_ids]

print("Input IDs:", input_ids)
print("\nEmbedding output:")
print(lookup_result)
print("Output shape:", lookup_result.shape)
print("\nRows selected directly:")
print(selected_rows)
print("All values equal:", torch.equal(lookup_result, selected_rows))

Input IDs: tensor([2, 3, 5, 1])

Embedding output:
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)
Output shape: torch.Size([4, 3])

Rows selected directly:
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<IndexBackward0>)
All values equal: True


The output shape is `[4, 3]`: four requested token IDs, each represented by three learned numbers.

## 6. GPT-2-scale embedding-matrix dimensions

We only calculate the dimensions and parameter count—we do not allocate this large matrix.

In [12]:
gpt2_vocab_size = 50_257
gpt2_embedding_dim = 768
gpt2_embedding_parameters = gpt2_vocab_size * gpt2_embedding_dim

print(f"Embedding matrix shape = {gpt2_vocab_size:,} x {gpt2_embedding_dim}")
print(f"Number of parameters = {gpt2_embedding_parameters:,}")
print(f"Approximately {gpt2_embedding_parameters / 1_000_000:.1f} million parameters")

Embedding matrix shape = 50,257 x 768
Number of parameters = 38,597,376
Approximately 38.6 million parameters


There are 50,257 rows—one for each vocabulary token—and 768 learned values in each row. All approximately 38.6 million values are trainable model parameters.

## 7. Watch embeddings learn in one training step

We now create the smallest useful next-token model—still **not a Transformer**:

```text
token ID
-> nn.Embedding
-> nn.Linear
-> next-token logits
```

For `the cat eats`, the shifted tasks are `the -> cat` and `cat -> eats`.

In [29]:
training_vocabulary = {"the": 0, "cat": 1, "eats": 2}
training_input_ids = torch.tensor([0, 1])   # the, cat
training_target_ids = torch.tensor([1, 2])  # cat, eats

print("Input IDs: ", training_input_ids.tolist(), "= ['the', 'cat']")
print("Target IDs:", training_target_ids.tolist(), "= ['cat', 'eats']")

Input IDs:  [0, 1] = ['the', 'cat']
Target IDs: [1, 2] = ['cat', 'eats']


In [14]:
torch.manual_seed(7)
toy_embedding = nn.Embedding(num_embeddings=3, embedding_dim=4)
output_layer = nn.Linear(in_features=4, out_features=3)
optimizer = torch.optim.SGD(
    list(toy_embedding.parameters()) + list(output_layer.parameters()),
    lr=0.2,
)

embeddings_before = toy_embedding.weight.detach().clone()
print("Embedding rows before training:")
print(embeddings_before)

Embedding rows before training:
tensor([[-0.1468,  0.7861,  0.9468, -1.1143],
        [ 1.6908, -0.8948, -0.3556,  1.2324],
        [ 0.1382, -1.6822,  0.3177,  0.1328]])


One training step consists of a forward pass, cross-entropy loss, backpropagation, and an optimizer update.

In [15]:
optimizer.zero_grad()

embedded_inputs = toy_embedding(training_input_ids)
next_token_logits = output_layer(embedded_inputs)
loss = F.cross_entropy(next_token_logits, training_target_ids)

loss.backward()
embedding_gradients = toy_embedding.weight.grad.detach().clone()
optimizer.step()

embeddings_after = toy_embedding.weight.detach().clone()
embedding_changes = embeddings_after - embeddings_before

print("Loss:", loss.item())
print("\nEmbedding gradients:")
print(embedding_gradients)
print("\nEmbedding rows after one update:")
print(embeddings_after)
print("\nChange in each row:")
print(embedding_changes)
print("\nAt least one embedding changed:", not torch.equal(embeddings_before, embeddings_after))

assert embedding_gradients[0].abs().sum() > 0
assert embedding_gradients[1].abs().sum() > 0
assert not torch.equal(embeddings_before, embeddings_after)

Loss: 1.1797170639038086

Embedding gradients:
tensor([[-0.0773,  0.1771, -0.0098,  0.0292],
        [ 0.0184, -0.0931,  0.0371, -0.0933],
        [ 0.0000,  0.0000,  0.0000,  0.0000]])

Embedding rows after one update:
tensor([[-0.1313,  0.7507,  0.9488, -1.1202],
        [ 1.6871, -0.8762, -0.3631,  1.2510],
        [ 0.1382, -1.6822,  0.3177,  0.1328]])

Change in each row:
tensor([[ 0.0155, -0.0354,  0.0020, -0.0058],
        [-0.0037,  0.0186, -0.0074,  0.0187],
        [ 0.0000,  0.0000,  0.0000,  0.0000]])

At least one embedding changed: True


The embedding vectors were not manually designed. They started randomly. Next-token prediction produced a loss, backpropagation calculated gradients, and the optimizer changed the embedding weights.

Only `the` and `cat` appeared as **input tokens**, so their embedding rows received gradients in this tiny example. `eats` appeared only as a target; it affects the output-layer gradient but its input embedding row is unused and remains unchanged in this one step.

## 8. How semantic meaning can emerge

GPT is not explicitly told that cat and dog are similar or that cat and banana are different. During large-scale next-token training it repeatedly encounters related contexts:

```text
The cat ate its food.
The dog ate its food.

The cat is an animal.
The dog is an animal.

She has a pet cat.
She has a pet dog.
```

Similar tokens repeatedly participate in similar prediction problems. Across enormous amounts of text and optimizer updates, useful relationships can gradually emerge in the learned representations. **Semantic structure emerges as a consequence of optimizing next-token prediction**, not because someone manually assigns a meaning to each dimension.

## 9. Static token embedding versus contextual representation

Consider:

```text
I deposited money in the bank.
We sat beside the river bank.
```

The `bank` token initially retrieves the same learned token-embedding row in both sentences. After Transformer self-attention processes the surrounding tokens, its hidden or contextual representation can differ:

```text
bank token
-> same initial token embedding

money + deposited + bank
-> Transformer
-> financial contextual representation

river + bank
-> Transformer
-> river-edge contextual representation
```

A token embedding is therefore not the token's final context-dependent representation. No Transformer is implemented in this notebook.

## 10. `nn.Embedding` versus one-hot matrix multiplication

Mathematically, a one-hot row multiplied by an embedding weight matrix selects one matrix row. We can prove that this produces the same values as `nn.Embedding`.

In [16]:
equivalence_vocab_size = 4
equivalence_embedding_dim = 5
ids = torch.tensor([2, 3, 1])

torch.manual_seed(99)
equivalence_embedding = nn.Embedding(
    equivalence_vocab_size,
    equivalence_embedding_dim,
)

embedding_output = equivalence_embedding(ids)
one_hot_ids = F.one_hot(ids, num_classes=equivalence_vocab_size).float()
matrix_multiplication_output = one_hot_ids @ equivalence_embedding.weight

print("IDs:", ids)
print("\nOne-hot rows:")
print(one_hot_ids)
print("\nnn.Embedding output:")
print(embedding_output)
print("\nOne-hot @ same weight matrix:")
print(matrix_multiplication_output)
print("\nOutputs equal:", torch.allclose(embedding_output, matrix_multiplication_output))

assert torch.allclose(embedding_output, matrix_multiplication_output)

IDs: tensor([2, 3, 1])

One-hot rows:
tensor([[0., 0., 1., 0.],
        [0., 0., 0., 1.],
        [0., 1., 0., 0.]])

nn.Embedding output:
tensor([[-1.2423e-03, -4.9816e-01, -3.4563e-01, -4.9046e-01,  6.1284e-01],
        [ 3.0975e+00,  1.2142e+00, -5.1517e-01,  1.1804e+00, -4.8238e-01],
        [ 8.8404e-02,  5.3131e-02, -7.9238e-01, -6.9572e-01,  8.0299e-01]],
       grad_fn=<EmbeddingBackward0>)

One-hot @ same weight matrix:
tensor([[-1.2423e-03, -4.9816e-01, -3.4563e-01, -4.9046e-01,  6.1284e-01],
        [ 3.0975e+00,  1.2142e+00, -5.1517e-01,  1.1804e+00, -4.8238e-01],
        [ 8.8404e-02,  5.3131e-02, -7.9238e-01, -6.9572e-01,  8.0299e-01]],
       grad_fn=<MmBackward0>)

Outputs equal: True


Both calculations produce the same vectors. `nn.Embedding` is preferred because it directly retrieves the required rows instead of constructing enormous mostly-zero one-hot vectors and performing unnecessary multiplications. That efficiency matters when the vocabulary contains tens of thousands of tokens.

## 11. Connection to actual LLM training

```text
Raw text
-> Tokenizer
-> Token IDs [4, 20, 91, ...]
-> Embedding lookup
-> Vectors
-> Add positional information
-> Transformer
-> Next-token predictions
-> Cross-entropy loss
-> Backpropagation
-> Gradients for:
     - attention weights
     - feed-forward weights
     - output weights
     - TOKEN EMBEDDING WEIGHTS
-> Optimizer update
-> Repeat over enormous amounts of text
```

**Token embeddings are not a preprocessing result that GPT receives already trained. The token embedding matrix is itself a set of trainable model parameters. It starts with random values and is learned jointly with the rest of the LLM through next-token prediction and backpropagation.**

## 12. Final recap

```text
Token -> tokenizer -> token ID
Token ID -> embedding-matrix row
Vocabulary size -> number of embedding rows
Embedding dimension -> numbers per token vector
Embedding matrix -> trainable weights
Initially -> random
During LLM training -> updated by backpropagation
nn.Embedding -> efficient lookup table
Word2Vec/GloVe demo -> vectors can capture semantic relationships
GPT embeddings -> learned jointly with the full LLM
Token embedding != final contextual representation
```